# 실습 3-1. 도장 외관 이상탐지

가상 공장의 **도장 외관 검사 셀**에서 찍은 패널 이미지로, **정상 이미지만 학습해서** 결함을 찾음.

- 불량 이미지를 모으기 어려운 현장에서 쓰는 방법임 (정상은 많고 불량은 드묾)
- 원리: 정상 이미지의 화소별 평균과 흔들림(표준편차)을 기억해 두고, 새 이미지가 거기서 얼마나 벗어났는지 봄
- 조건: 조명 · 배경 · 위치가 **고정**되어 있어야 함

- **기본 트랙**: 셀을 차례로 실행하고 임계값별 과검 · 미검 표를 워크북에 옮겨 적음
- **심화 트랙**: `# 심화` 표시가 있는 값을 바꿔 AUROC 0.90 이상을 확인하고 과검 · 미검의 균형점을 정함

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists():      # 저장소 최상위 폴더 찾기
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common.plot_style import use_korean_font
use_korean_font()

DATA = ROOT / "data" / "session3"
OUT = DATA / "work"                        # 내 산출물이 저장되는 곳
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

## 1. 데이터 확인

| 구분 | 장수 | 내용 |
|---|---|---|
| train | 240 | 양품만 (학습용) |
| test | 120 | 양품 60 + 결함 60 (핀홀 · 흐름 · 이물 · 색차 각 15) |
| shift | 30 | 양품이지만 촬영 조건이 바뀐 것 (조명 각도 · 밝기 · 배경판) |

In [ ]:
images = np.load(DATA / "paint_images.npz")["images"]          # [390, 192, 192, 3]
labels = pd.read_csv(DATA / "paint_labels.csv")
KIND_KO = {"none": "양품", "pinhole": "핀홀", "sag": "흐름", "dust": "이물", "tint": "색차"}
print(images.shape)
labels.groupby(["split", "kind"]).size().unstack(fill_value=0)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3.2))
for ax, kind in zip(axes, KIND_KO):
    i = labels[(labels.split == "test") & (labels.kind == kind)].index[0]
    ax.imshow(images[i]); ax.set_title(KIND_KO[kind]); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. 정상 학습

양품 240장의 화소별 평균과 표준편차를 구함. 이것이 "정상의 모습"임.

In [ ]:
from common.anomaly import PixelModel, prepare

t0 = time.time()
X = np.stack([prepare(im) for im in images])                 # 96 x 96 으로 줄임
is_train = (labels.split == "train").values
model = PixelModel().fit(X[is_train])
print(f"학습 시간 {time.time() - t0:.1f}초  (목표: 노트북 CPU 5분 이내)")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
axes[0].imshow(model.mean); axes[0].set_title("정상의 평균 모습"); axes[0].axis("off")
im = axes[1].imshow(model.std.max(axis=-1), cmap="viridis"); axes[1].set_title("정상에서도 흔들리는 곳"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046); plt.tight_layout(); plt.show()

> 패널 가장자리는 시편 위치가 조금씩 달라서 정상에서도 흔들림. 그런 곳은 조금 달라도 이상으로 보지 않음.

## 3. 결함 영역 검출

새 이미지가 정상에서 벗어난 정도를 화소마다 구해 지도로 그림.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for j, kind in enumerate(KIND_KO):
    i = labels[(labels.split == "test") & (labels.kind == kind)].index[1]
    amap = model.anomaly_map(X[i])
    axes[0, j].imshow(images[i]); axes[0, j].set_title(KIND_KO[kind]); axes[0, j].axis("off")
    axes[1, j].imshow(amap, cmap="inferno", vmin=0, vmax=10)
    axes[1, j].set_title(f"이상 점수 {model.score(X[i]):.1f}"); axes[1, j].axis("off")
plt.tight_layout(); fig.savefig(OUT / "anomaly_maps.png", dpi=120); plt.show()

## 4. 이상 점수와 AUROC

이미지 한 장의 이상 점수 = 지도에서 가장 높은 부분의 평균. 양품과 결함의 점수가 얼마나 갈리는지 봄.

In [ ]:
from sklearn.metrics import roc_auc_score

labels["score"] = [model.score(x) for x in X]
test = labels[labels.split == "test"]
auroc = roc_auc_score(test["is_defect"], test["score"])
print(f"AUROC {auroc:.3f}   (목표 0.90 이상)")

fig, ax = plt.subplots(figsize=(10, 3.4))
bins = np.linspace(0, 13, 53)
for kind, color in zip(KIND_KO, ["#2471a3", "#7d3c98", "#d68910", "#566573", "#1e8449"]):
    ax.hist(test.loc[test.kind == kind, "score"], bins=bins, alpha=0.75, label=KIND_KO[kind], color=color)
ax.set_xlabel("이상 점수"); ax.set_ylabel("장수"); ax.legend(ncol=5); ax.set_title("시험 이미지의 이상 점수 분포")
plt.tight_layout(); fig.savefig(OUT / "score_hist.png", dpi=120); plt.show()
test.groupby("kind")["score"].agg(["min", "median", "max"]).round(1).rename(index=KIND_KO)

> 핀홀 · 흐름은 점수가 뚜렷하게 높음. **색차**는 양품과 거의 붙어 있음 → 임계값을 어디에 두느냐에 따라 잡히기도 놓치기도 함.

## 5. 임계값과 과검 · 미검

In [ ]:
rows = []
for th in [1.0, 1.5, 2.0, 2.2, 2.3, 2.4, 2.5, 3.0, 4.0, 6.0, 8.0]:
    ng = test["score"] >= th
    rows.append({"임계값": th, "과검(양품→불량)": int((ng & (test.is_defect == 0)).sum()),
                 "미검(불량→양품)": int((~ng & (test.is_defect == 1)).sum())})
result = pd.DataFrame(rows).set_index("임계값")
result["과검률(%)"] = (result["과검(양품→불량)"] / (test.is_defect == 0).sum() * 100).round(1)
result["미검률(%)"] = (result["미검(불량→양품)"] / (test.is_defect == 1).sum() * 100).round(1)
result.to_csv(OUT / "anomaly_result.csv", encoding="utf-8-sig")
result

In [ ]:
THRESHOLD = 2.5                         # 심화: 과검과 미검의 균형점을 직접 정함
ng = test["score"] >= THRESHOLD
print(f"임계값 {THRESHOLD}:  과검 {int((ng & (test.is_defect == 0)).sum())}장,  미검 {int((~ng & (test.is_defect == 1)).sum())}장")
(test.assign(검출=ng).groupby("kind")["검출"].mean() * 100).round(0).rename(index=KIND_KO).rename("불량으로 판정한 비율(%)").to_frame()

> 임계값을 낮추면 색차까지 잡지만 양품을 불량으로 보내는 과검이 늘고, 높이면 과검은 줄지만 미검이 늚.
> 가상 공장의 임계값 슬라이더로 같은 현상을 확인함.

## 6. 촬영 조건이 바뀌면

같은 **양품**을 조명 각도 · 밝기 · 배경판을 바꿔 찍은 30장임. 결함이 없는데도 점수가 어떻게 되는지 봄.

In [ ]:
shift = labels[labels.split == "shift"].copy()
COND_KO = {'{"angle": 25.0}': "조명 각도 45→25도", '{"level": 1.5}': "조명 밝기 1.5배", '{"backdrop": false}': "배경판 없음"}
shift["조건"] = shift["condition"].map(COND_KO)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
i0 = labels[(labels.split == "test") & (labels.kind == "none")].index[0]
axes[0].imshow(images[i0]); axes[0].set_title(f"기준 조건  (점수 {labels.score[i0]:.1f})"); axes[0].axis("off")
for ax, (name, g) in zip(axes[1:], shift.groupby("조건", sort=False)):
    ax.imshow(images[g.index[0]]); ax.set_title(f"{name}  (점수 {g.score.iloc[0]:.1f})"); ax.axis("off")
plt.tight_layout(); fig.savefig(OUT / "shift.png", dpi=120); plt.show()
out = shift.groupby("조건", sort=False)["score"].agg(["median", "min", "max"]).round(1)
out["불량 판정(장)"] = shift.groupby("조건", sort=False)["score"].apply(lambda s: int((s >= THRESHOLD).sum()))
out

> 결함이 하나도 없는데 대부분 불량으로 판정됨. 이 방법은 **촬영 조건을 고정**해야만 쓸 수 있음.
> 현장에서 조명을 교체하거나 지그 위치를 옮겼다면 정상 이미지를 다시 모아 **재학습**해야 함.

## 7. 저장

In [ ]:
model.save(OUT / "pixel_model.npz")       # 가상 공장의 "모델 다시 읽기" 버튼이 이 파일을 읽음
print("저장:", [p.name for p in sorted(OUT.glob("*")) if p.name.startswith(("anomaly", "pixel", "score", "shift"))])

### 워크북에 옮겨 적기

1. AUROC 값과 학습 시간
2. 임계값별 과검 · 미검 표 (5절) 와 내가 정한 임계값
3. 색차를 잡으려면 임계값 외에 무엇을 바꿔야 할지 한 문장

### 심화

- `PixelModel` 의 표준편차 하한(0.012)을 바꾸면: `model.std = X[is_train].std(axis=0) + 0.005` 후 4절부터 다시 실행
- 정상 이미지를 60장만 써서 학습하면 AUROC 와 과검이 어떻게 달라지는지: `PixelModel().fit(X[is_train][:60])`